<a href="https://colab.research.google.com/github/ZAKARIAE48CHELLE/reqmind-ai/blob/main/models/ReqMind_AI_Step3_DeBERTa_FineTuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ReqMind AI — Model Optimization: Step 3 (Repaired & Stabilized)
## Fine-Tuning `microsoft/deberta-v3-small` for FR / NFR Classification

### Diagnostics & Root-Cause Resolution:
1. **Root-Cause of NaN Loss:** The official Hugging Face `microsoft/deberta-v3-small` `config.json` specifies `"torch_dtype": "float16"`. Loading without explicit `torch_dtype=torch.float32` causes DeBERTa's disentangled attention position embeddings to exceed the FP16 dynamic range (>65,504), resulting in attention overflow, NaN softmax, NaN loss, and weight collapse into majority-only predictions (65.68% accuracy, 0% NFR-F1).
2. **The Fix:** Enforce `torch_dtype=torch.float32`, `fp16=False`, `bf16=False`, and `max_grad_norm=1.0` during initialization and training.
3. **Fail-Fast NaN Protection:** Added `NanDetectorCallback` to immediately halt on any non-finite loss.
4. **Full Model Weight Packaging:** Validates that `model.safetensors` (~540 MB uncompressed, ~150 MB compressed) is fully preserved in the export archive.

## 1. Environment & Hardware Verification (GPU Check)
Verify CUDA device availability in Colab (**Runtime > Change runtime type > T4 GPU**).

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:     {torch.cuda.get_device_name(0)}")
    print(f"Device VRAM:     {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("[*] Running on CPU. For fast training, enable GPU via Runtime -> Change runtime type -> T4 GPU.")

## 2. Install Required Dependencies

In [ ]:
!pip install -q transformers datasets accelerate evaluate scikit-learn matplotlib seaborn

## 3. Load Audited Splits (Direct Upload or Local Path)
Upload the 3 small split CSVs (~600 KB total) if running in Google Colab.

In [ ]:
import os, shutil
from pathlib import Path

target_dir = Path("Data/splits_step3")
target_dir.mkdir(parents=True, exist_ok=True)

# If running in Google Colab and files are not present, trigger upload
if not (target_dir / "deberta_train.csv").exists():
    try:
        from google.colab import files
        print("[*] Please select and upload the 3 CSV files from your local Data/splits_step3/:")
        print("    - deberta_train.csv")
        print("    - deberta_val.csv")
        print("    - deberta_test_sealed.csv")
        uploaded = files.upload()
        for fname in uploaded.keys():
            shutil.move(fname, str(target_dir / fname))
        print(f"[*] Successfully uploaded {len(uploaded)} files to {target_dir}/")
    except ImportError:
        print("[*] Running locally; checking Data/splits_step3/...")

print("[*] Verified split files in:", target_dir.resolve())
for f in sorted(target_dir.glob("*.csv")):
    print(f"    - {f.name} ({f.stat().st_size / 1024:.1f} KB)")

## 4. Dataset Inspection & Integrity Verification

In [ ]:
import pandas as pd
import numpy as np

train_df = pd.read_csv("Data/splits_step3/deberta_train.csv")
val_df   = pd.read_csv("Data/splits_step3/deberta_val.csv")
test_df  = pd.read_csv("Data/splits_step3/deberta_test_sealed.csv")

print(f"[*] Training Sub-Split:   {len(train_df):,} rows (FR: {(train_df.label=='FR').sum():,}, NFR: {(train_df.label=='NFR').sum():,})")
print(f"[*] Validation Sub-Split: {len(val_df):,} rows   (FR: {(val_df.label=='FR').sum():,}, NFR: {(val_df.label=='NFR').sum():,})")
print(f"[*] Sealed Test Set:      {len(test_df):,} rows  (FR: {(test_df.label=='FR').sum():,}, NFR: {(test_df.label=='NFR').sum():,}) [SEALED]")

# Zero leakage assertion
tr_keys = set(train_df.text.str.lower().str.strip())
val_keys = set(val_df.text.str.lower().str.strip())
test_keys = set(test_df.text.str.lower().str.strip())

assert len(tr_keys.intersection(val_keys)) == 0, "Data leakage between Train and Validation!"
assert len(tr_keys.intersection(test_keys)) == 0, "Data leakage between Train and Test!"
assert len(val_keys.intersection(test_keys)) == 0, "Data leakage between Validation and Test!"
print("[*] Data integrity check: PASSED. Zero exact text overlap across partitions.")

## 5. Tokenization with DeBERTa-v3-small Tokenizer

In [ ]:
from transformers import AutoTokenizer, DataCollatorWithPadding
from datasets import Dataset

MODEL_ID = "microsoft/deberta-v3-small"
LABEL2ID = {"FR": 0, "NFR": 1}
ID2LABEL = {0: "FR", 1: "NFR"}

print(f"[*] Loading tokenizer: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=96)

train_df["target"] = train_df["label"].map(LABEL2ID)
val_df["target"]   = val_df["label"].map(LABEL2ID)

train_ds = Dataset.from_pandas(train_df[["sample_id", "text", "target"]].rename(columns={"target": "label"}))
val_ds   = Dataset.from_pandas(val_df[["sample_id", "text", "target"]].rename(columns={"target": "label"}))

tokenized_train = train_ds.map(tokenize_batch, batched=True, remove_columns=["text", "sample_id"])
tokenized_val   = val_ds.map(tokenize_batch, batched=True, remove_columns=["text", "sample_id"])
data_collator   = DataCollatorWithPadding(tokenizer=tokenizer)

print(f"[*] Tokenization complete. Training size: {len(tokenized_train)}, Validation size: {len(tokenized_val)}")

## 6. Evaluation Metrics & Fail-Safe Callbacks
- `MetricHistoryCallback`: Logs loss and per-class metrics each epoch.
- `NanDetectorCallback`: Fail-fast check that raises immediately if non-finite loss is encountered.

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from transformers import TrainerCallback

class NanDetectorCallback(TrainerCallback):
    """Fail-fast callback that immediately halts if any training or validation loss is non-finite."""
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None and "loss" in logs:
            loss = logs["loss"]
            if not np.isfinite(loss):
                raise ValueError(f"FATAL: Non-finite training loss ({loss}) detected at step {state.global_step}! Aborting.")

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics is not None and "eval_loss" in metrics:
            eval_loss = metrics["eval_loss"]
            if not np.isfinite(eval_loss):
                raise ValueError(f"FATAL: Non-finite validation loss ({eval_loss}) detected at step {state.global_step}! Aborting.")

class MetricHistoryCallback(TrainerCallback):
    def __init__(self):
        super().__init__()
        self.history = []

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics:
            self.history.append({
                "epoch": state.epoch,
                "step": state.global_step,
                "eval_loss": metrics.get("eval_loss"),
                "eval_accuracy": metrics.get("eval_accuracy"),
                "eval_macro_f1": metrics.get("eval_macro_f1"),
                "eval_f1_fr": metrics.get("eval_f1_fr"),
                "eval_f1_nfr": metrics.get("eval_f1_nfr"),
            })

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro")
    f1_fr = f1_score(labels, preds, pos_label=0)
    f1_nfr = f1_score(labels, preds, pos_label=1)
    prec_fr = precision_score(labels, preds, pos_label=0, zero_division=0)
    rec_fr = recall_score(labels, preds, pos_label=0, zero_division=0)
    prec_nfr = precision_score(labels, preds, pos_label=1, zero_division=0)
    rec_nfr = recall_score(labels, preds, pos_label=1, zero_division=0)
    return {
        "accuracy": round(float(acc), 4),
        "macro_f1": round(float(macro_f1), 4),
        "f1_fr": round(float(f1_fr), 4),
        "precision_fr": round(float(prec_fr), 4),
        "recall_fr": round(float(rec_fr), 4),
        "f1_nfr": round(float(f1_nfr), 4),
        "precision_nfr": round(float(prec_nfr), 4),
        "recall_nfr": round(float(rec_nfr), 4),
    }

## 7. Model Initialization in Pure FP32 & Trainer Configuration
- Explicit `torch_dtype=torch.float32` prevents FP16 attention overflow.
- `fp16=False` and `bf16=False` ensures numerical stability in all gradient updates.
- `max_grad_norm=1.0` clips gradient bursts.
- `learning_rate=1e-5` ensures conservative, stable convergence.

In [ ]:
import torch
import inspect
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback

# Load model strictly in float32 to override DeBERTa-v3 config.json's float16 default
print(f"[*] Loading {MODEL_ID} with explicit torch_dtype=torch.float32...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    torch_dtype=torch.float32,
)

# Verify all parameters are float32 and finite
for name, p in model.named_parameters():
    assert p.dtype == torch.float32, f"Parameter {name} has dtype {p.dtype}, expected float32!"
    assert torch.isfinite(p).all(), f"Parameter {name} contains non-finite initial values!"
print("[*] Parameter validation: ALL PARAMETERS ARE FLOAT32 AND FINITE.")

metric_logger = MetricHistoryCallback()
nan_detector = NanDetectorCallback()
early_stopping = EarlyStoppingCallback(early_stopping_patience=2)

training_args_kwargs = {
    "output_dir": "checkpoints_deberta",
    "save_strategy": "epoch",
    "learning_rate": 1e-5,
    "per_device_train_batch_size": 16,
    "per_device_eval_batch_size": 32,
    "num_train_epochs": 5,
    "weight_decay": 0.01,
    "max_grad_norm": 1.0,
    "warmup_steps": 100,
    "metric_for_best_model": "macro_f1",
    "greater_is_better": True,
    "load_best_model_at_end": True,
    "fp16": False,
    "bf16": False,
    "logging_strategy": "epoch",
    "save_total_limit": 2,
    "report_to": "none",
    "seed": 42,
}

if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters:
    training_args_kwargs["eval_strategy"] = "epoch"
else:
    training_args_kwargs["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**training_args_kwargs)

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": tokenized_train,
    "eval_dataset": tokenized_val,
    "data_collator": data_collator,
    "compute_metrics": compute_metrics,
    "callbacks": [early_stopping, metric_logger, nan_detector],
}

if "processing_class" in inspect.signature(Trainer.__init__).parameters:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)
print("[*] Trainer successfully initialized in stable FP32 mode.")

## 8. Execute DeBERTa-v3-small Fine-Tuning

In [ ]:
import time

print("[*] Commencing DeBERTa-v3-small training (FP32 numerical stability)... ")
t0 = time.perf_counter()
train_result = trainer.train()
duration = round(time.perf_counter() - t0, 2)
print(f"[*] Fine-tuning finished in {duration} seconds ({duration/60:.2f} minutes).")
print(f"[*] Final Training Loss: {train_result.training_loss:.4f} (Finite: {np.isfinite(train_result.training_loss)})")
assert np.isfinite(train_result.training_loss), "FATAL: Training loss is non-finite!"

## 9. Comprehensive Validation Evaluation & Metric Extraction

In [ ]:
eval_results = trainer.evaluate()
print("=" * 60)
print("DEBERTA-V3-SMALL VALIDATION RESULTS (N=950)")
print("=" * 60)
print(f"Accuracy:  {eval_results['eval_accuracy']*100:.2f}%")
print(f"Macro-F1:  {eval_results['eval_macro_f1']*100:.2f}%")
print(f"FR-F1:     {eval_results['eval_f1_fr']*100:.2f}% (Precision: {eval_results['eval_precision_fr']*100:.2f}%, Recall: {eval_results['eval_recall_fr']*100:.2f}%)")
print(f"NFR-F1:    {eval_results['eval_f1_nfr']*100:.2f}% (Precision: {eval_results['eval_precision_nfr']*100:.2f}%, Recall: {eval_results['eval_recall_nfr']*100:.2f}%)")
print(f"Eval Loss: {eval_results['eval_loss']:.4f}")

assert np.isfinite(eval_results['eval_loss']), "FATAL: Validation loss is non-finite!"
assert eval_results['eval_f1_nfr'] > 0.50, f"Degenerate model! NFR-F1 is {eval_results['eval_f1_nfr']}"

## 10. Confusion Matrix & Training Dynamics Visualization

In [ ]:
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# Out-of-fold predictions on validation set
predictions = trainer.predict(tokenized_val)
logits = predictions.predictions
assert np.isfinite(logits).all(), "Logits contain NaN/Inf!"

probs = torch.softmax(torch.as_tensor(logits), dim=-1).numpy()
pred_indices = np.argmax(probs, axis=1)
pred_labels = [ID2LABEL[i] for i in pred_indices]
confidences = np.max(probs, axis=1)

# Verify the model predicts BOTH classes
unique_preds = set(pred_labels)
print(f"[*] Predicted class distribution: FR: {(pred_indices == 0).sum()}, NFR: {(pred_indices == 1).sum()}")
assert len(unique_preds) == 2, f"Model collapsed to single class: {unique_preds}!"

cm = confusion_matrix(val_df["label"], pred_labels, labels=["FR", "NFR"])

# 1. Plot Confusion Matrix
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Purples)
ax.figure.colorbar(im, ax=ax)
ax.set(
    xticks=[0, 1], yticks=[0, 1],
    xticklabels=["FR", "NFR"], yticklabels=["FR", "NFR"],
    ylabel="Ground Truth", xlabel="Predicted Label",
    title=f"DeBERTa-v3-small Validation Confusion Matrix\nAccuracy: {eval_results['eval_accuracy']*100:.2f}% | Macro-F1: {eval_results['eval_macro_f1']*100:.2f}%"
)
thresh = cm.max() / 2.0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, format(cm[i, j], "d"),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black",
                fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("step3_deberta_val_confusion_matrix.png", dpi=200)
plt.show()

# 2. Plot Training and Validation Curves
if metric_logger.history:
    history_df = pd.DataFrame(metric_logger.history)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

    ax1.plot(history_df["epoch"], history_df["eval_loss"], marker="o", color="crimson", linewidth=2, label="Validation Loss")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.set_title("Validation Loss Trajectory")
    ax1.grid(True, linestyle=":", alpha=0.6)
    ax1.legend()

    ax2.plot(history_df["epoch"], history_df["eval_macro_f1"], marker="s", color="indigo", linewidth=2, label="Validation Macro-F1")
    ax2.plot(history_df["epoch"], history_df["eval_f1_nfr"], marker="^", color="darkorange", linestyle="--", label="Validation NFR-F1")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("F1 Score")
    ax2.set_title("Validation F1 Performance Trajectory")
    ax2.grid(True, linestyle=":", alpha=0.6)
    ax2.legend()

    plt.tight_layout()
    plt.savefig("step3_deberta_training_curves.png", dpi=200)
    plt.show()

## 11. Inference Latency Benchmark

In [ ]:
import time
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval()
benchmark_texts = val_df["text"].iloc[:100].tolist()

with torch.no_grad():
    enc = tokenizer(benchmark_texts, padding=True, truncation=True, max_length=96, return_tensors="pt").to(device)
    # Warmup
    _ = model(**enc)
    t_start = time.perf_counter()
    _ = model(**enc)
    latency_ms = ((time.perf_counter() - t_start) / 100.0) * 1000.0

print(f"[*] Batched Inference Latency: {latency_ms:.3f} ms per requirement on {device}")

## 12. Fair Comparison Against Baseline & Optimized SVM (Live Recomputed)
We train and evaluate both the Baseline and Step 2 Optimized SVM models on the exact same 3,798 train / 950 validation splits to ensure an empirical, zero-fabrication comparison.

In [ ]:
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

print("[*] Training Baseline SVM on identical 3,798 training samples...")
base_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
    ("clf", LinearSVC(C=1.0, random_state=42, max_iter=2000)),
])
base_pipe.fit(train_df["text"], train_df["label"])
base_val_preds = base_pipe.predict(val_df["text"])

print("[*] Training Step 2 Optimized SVM on identical 3,798 training samples...")
opt_pipe = Pipeline([
    ("features", FeatureUnion([
        ("word_tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ("char_tfidf", TfidfVectorizer(ngram_range=(3, 5), analyzer="char_wb", min_df=3, sublinear_tf=True)),
    ])),
    ("clf", LinearSVC(C=1.0, class_weight="balanced", random_state=42, max_iter=3000)),
])
opt_pipe.fit(train_df["text"], train_df["label"])
opt_val_preds = opt_pipe.predict(val_df["text"])

# Live Recomputed Table
comparison_table = pd.DataFrame([
    {
        "Model": "Baseline Linear SVM (Word TF-IDF)",
        "Representation": "Word n-grams (1,2)",
        "Val Accuracy": f"{accuracy_score(val_df['label'], base_val_preds)*100:.2f}%",
        "Val Macro-F1": f"{f1_score(val_df['label'], base_val_preds, average='macro')*100:.2f}%",
        "FR-F1": f"{f1_score(val_df['label'], base_val_preds, pos_label='FR')*100:.2f}%",
        "NFR-F1": f"{f1_score(val_df['label'], base_val_preds, pos_label='NFR')*100:.2f}%",
    },
    {
        "Model": "Step 2 Optimized SVM (Word+Char Balanced)",
        "Representation": "Word (1,2) + Char (3,5)",
        "Val Accuracy": f"{accuracy_score(val_df['label'], opt_val_preds)*100:.2f}%",
        "Val Macro-F1": f"{f1_score(val_df['label'], opt_val_preds, average='macro')*100:.2f}%",
        "FR-F1": f"{f1_score(val_df['label'], opt_val_preds, pos_label='FR')*100:.2f}%",
        "NFR-F1": f"{f1_score(val_df['label'], opt_val_preds, pos_label='NFR')*100:.2f}%",
    },
    {
        "Model": "microsoft/deberta-v3-small (Fine-Tuned)",
        "Representation": "Disentangled Transformer (44M params)",
        "Val Accuracy": f"{eval_results['eval_accuracy']*100:.2f}%",
        "Val Macro-F1": f"{eval_results['eval_macro_f1']*100:.2f}%",
        "FR-F1": f"{eval_results['eval_f1_fr']*100:.2f}%",
        "NFR-F1": f"{eval_results['eval_f1_nfr']*100:.2f}%",
    },
])

import IPython.display as disp
disp.display(comparison_table)

## 13. Export Validation Errors Log

In [ ]:
val_errors_df = val_df.copy()
val_errors_df["predicted_label"] = pred_labels
val_errors_df["confidence"] = np.round(confidences, 4)
val_errors_df["prob_FR"] = np.round(probs[:, 0], 4)
val_errors_df["prob_NFR"] = np.round(probs[:, 1], 4)
val_errors_df["is_correct"] = val_errors_df["label"] == val_errors_df["predicted_label"]

errors_only = val_errors_df[~val_errors_df["is_correct"]].reset_index(drop=True)
errors_only.to_csv("step3_deberta_val_errors.csv", index=False, encoding="utf-8")
print(f"[*] Exported {len(errors_only)} misclassifications to step3_deberta_val_errors.csv")
errors_only[["sample_id", "label", "predicted_label", "confidence", "text"]].head(10)

## 14. Save Model Artifacts, Packaging & Reload Validation

In [ ]:
import os, zipfile
from pathlib import Path

MODEL_EXPORT_DIR = "models/deberta_v3_small_fr_nfr"
os.makedirs(MODEL_EXPORT_DIR, exist_ok=True)

print(f"[*] Saving model and tokenizer to {MODEL_EXPORT_DIR}...")
trainer.save_model(MODEL_EXPORT_DIR)
tokenizer.save_pretrained(MODEL_EXPORT_DIR)

# Verify weights exist and have realistic size
exported_files = list(Path(MODEL_EXPORT_DIR).glob("*"))
weight_files = [f for f in exported_files if "safetensors" in f.name or "bin" in f.name]
assert len(weight_files) > 0, "FATAL: Model weights (model.safetensors) missing from export!"
for wf in weight_files:
    print(f"[*] Verified weights file: {wf.name} ({wf.stat().st_size / (1024*1024):.2f} MB)")

# 15. Reload Validation in Fresh Model Session
print("[*] Testing fresh reload from saved directory...")
reloaded_tok = AutoTokenizer.from_pretrained(MODEL_EXPORT_DIR)
reloaded_model = AutoModelForSequenceClassification.from_pretrained(MODEL_EXPORT_DIR)
reloaded_model.eval()

test_samples = [
    "The system shall allow users to upload PDF documents.",
    "The interface shall remain responsive on mobile screens."
]
with torch.no_grad():
    in1 = tokenizer(test_samples, padding=True, truncation=True, return_tensors="pt").to(model.device)
    in2 = reloaded_tok(test_samples, padding=True, truncation=True, return_tensors="pt").to(reloaded_model.device)
    diff = torch.abs(model(**in1).logits.cpu() - reloaded_model(**in2).logits.cpu()).max().item()
    print(f"[*] Max logit discrepancy between live and reloaded model: {diff:.6e}")
    assert diff < 1e-4, f"Reload mismatch: discrepancy is {diff}"
print("[*] Fresh reload validation: PASSED (identical predictions confirmed).")

# Create zip bundle cross-platform
zip_filename = "deberta_v3_small_reqmind.zip"
print(f"[*] Packaging deliverables into {zip_filename}...")
with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files_in_dir in os.walk(MODEL_EXPORT_DIR):
        for file in files_in_dir:
            filepath = os.path.join(root, file)
            arcname = os.path.relpath(filepath, start=".")
            zipf.write(filepath, arcname)
    for extra in ["step3_deberta_val_confusion_matrix.png", "step3_deberta_training_curves.png", "step3_deberta_val_errors.csv"]:
        if os.path.exists(extra):
            zipf.write(extra, extra)

zip_size_mb = os.path.getsize(zip_filename) / (1024 * 1024)
print(f"[*] Zip archive successfully created: {zip_filename} ({zip_size_mb:.2f} MB)")
assert zip_size_mb > 50, f"Zip file is unexpectedly small ({zip_size_mb:.2f} MB), weights were not packaged properly!"

# Automatic download trigger in Colab
try:
    from google.colab import files
    files.download(zip_filename)
    print("[*] Download dialog initiated for", zip_filename)
except ImportError:
    print("[*] Running locally; zip bundle saved at", zip_filename)